# Lesion Segmentation Hackathon: Starter Notebook

**Task:** each image is a coronal CT slice with **exactly one lesion** in the liver. Find and outline it **automatically**.

Run the cells from top to bottom. You write your method in **Section 4 (`segment`)**; scoring and the submission file are done for you.

| § | What happens |
|---|---|
| 1 | Setup |
| 2 | Loading the data |
| 3 | Viewing the data |
| 4 | **Your method: `segment()`** |
| 5 | Score your method on the development set |
| 6 | Make your submission file |


> In Colab: **File → Save a copy in Drive** first, so your changes are kept.

## 1  Setup
Run this cell once. In Colab it downloads the repository; on your own computer it uses the folder this notebook is in.

In [ ]:
REPO_URL = "https://github.com/rrr-uom-projects/MPiCRT-hackathon-segmentations.git"  

import os, sys, subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not Path("cases.csv").exists():
    if not Path("hackathon").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "hackathon"], check=True)
    os.chdir("hackathon")

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi

from score import dice, evaluate                               # scoring (score.py)
from submission_utils import write_submission, check_submission  # submission file (submission_utils.py)

cases = pd.read_csv(ROOT / "cases.csv")
dev_ids = cases.query("split == 'dev'").case_id.tolist()
test_ids = cases.query("split == 'test'").case_id.tolist()
print(f"{len(dev_ids)} development images, {len(test_ids)} test images")

## 2  Loading the data

| Folder | Contents |
|---|---|
| `data/dev/images/` | development images `dev_XX.npy` |
| `data/dev/lesion_masks/` | the true lesion masks ... |
| `data/test/images/` | the 10 test images. **No lesion masks**: ... |
| `data/dev/liver_masks/`, `data/test/liver_masks/` | optional liver masks for every image (1 = liver, 0 = elsewhere) |

- Every image is 256 × 256 pixels, 1.5 mm per pixel, in **Hounsfield units (HU)**.
- Rows run head → feet. Columns run the patient's right → left (the standard radiology view).

**Typical CT values (HU)**

| Tissue | HU |
|---|---|
| air | −1000 |
| fat | −100 to −50 |
| water | 0 |
| soft tissue / liver | +40 to +100 (higher after contrast injection) |
| bone | > +300 |

In [ ]:
def load_image(case_id):
    split = case_id.split("_")[0]                               # "dev" or "test"
    return np.load(ROOT / "data" / split / "images" / f"{case_id}.npy")

def load_mask(case_id):                                         # development images only
    return np.load(ROOT / "data" / "dev" / "lesion_masks" / f"{case_id}_mask.npy")

def load_liver(case_id):
    split = case_id.split("_")[0]
    path = ROOT / "data" / split / "liver_masks" / f"{case_id}_liver.npy"
    return np.load(path) if path.exists() else None


img, msk = load_image("dev_01"), load_mask("dev_01")
print("image:", img.shape, img.dtype, f"values {img.min()} … {img.max()} HU")
print("mask: ", msk.shape, msk.dtype, f"{msk.sum()} lesion pixels")

## 3  Viewing the data
CT values span thousands of HU, but a screen shows about 256 grey levels.
A **display window** maps one HU range to black → white; here −160 to +240 HU, which suits the abdomen.
The window only changes how the image looks; the data stay the same.

`show(ax, img, mask)` draws an image with an outline. Use it to explore.

In [ ]:
WIN = (-160, 240)

def show(ax, img, mask=None, title=""):
    ax.imshow(img, cmap="gray", vmin=WIN[0], vmax=WIN[1])
    if mask is not None and np.any(mask):
        ax.contour(mask, levels=[0.5], colors="red", linewidths=1)
    ax.set_title(title, fontsize=9)
    ax.axis("off")

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
show(ax[0], img, title="dev_01")
show(ax[1], img, msk, title="dev_01 with the true lesion outline")
plt.tight_layout(); plt.show()

**Before writing any code, explore the development set.** Some questions worth answering:
- What do the lesions look like compared with the liver around them?
- What HU values are inside a lesion? Inside normal liver? (`img[msk == 1]` gives the lesion pixels)
- How big are the lesions? Where are they?
- How different are the images from each other?

In [ ]:
# your exploration here

### Extra help: liver masks
Every image (development **and** test) also comes with a mask of the liver, in `data/dev/liver_masks/` and `data/test/liver_masks/`.
Load a few and look at them next to the CT images. How could they help you?

In [ ]:
# load and look at the liver masks here

## 4  Your method
`segment(img, liver)` gets one CT image (256 × 256, in HU) and, optionally, its liver mask (`None` if there isn't one). It must return a 256 × 256 mask: 1 = lesion, 0 = background.

You can write as many helper functions as you like in cells above; just call them from `segment()`.

The starter version keeps every pixel in a fixed HU range. It runs, but it scores badly. Make it better.

**Rule:** fully automatic. The same code must handle every image: no hand-drawn masks, no settings chosen for particular images.

In [ ]:
def segment(img, liver=None):
    # --- starter method: replace with your own ---
    LOW, HIGH = 0, 40                                    # HU range
    mask = (img > LOW) & (img < HIGH)
    return mask.astype(np.uint8)

## 5  Score your method on the development set
Runs `segment()` on every development image and compares it with the true mask using **Dice**
(1 = perfect overlap, 0 = no overlap). This is the same scoring used on the test set.

Re-run this cell every time you change `segment()`.

In [ ]:
scores = evaluate(segment, load_image, load_mask, dev_ids, load_liver=load_liver)
print("worst cases:", sorted(scores, key=scores.get)[:5])

## 6  Make your submission
1. Set your team name below.
2. Run the cell. It runs `segment()` on the 10 test images, writes `<team>.csv` and checks it.
3. Wait for **`OK — ready to submit`**, then send the CSV **and this notebook** to the organisers.

You can submit at most **3 times**; your best submission counts.

In [ ]:
TEAM = "team_name"                                  # <- your team name, no spaces

test_preds = {cid: segment(load_image(cid), load_liver(cid)) for cid in test_ids}
out_file = ROOT / f"{TEAM}.csv"
write_submission(test_preds, out_file)
check_submission(out_file, ROOT / "cases.csv")

if IN_COLAB:                                        # download straight away: Colab deletes files when it disconnects
    from google.colab import files
    files.download(str(out_file))